# Myanmar DTM — Cyclone Mocha (May 2023)

Cyclone Mocha made landfall near Sittwe, Rakhine State on **14 May 2023** — one of the most intense storms ever recorded in the Bay of Bengal.

This notebook demonstrates:
1. **DTM comparison** — Copernicus GLO-30 (30 m) vs GLO-90 (90 m) elevation data for the Rakhine State / Cyclone Mocha zone.
2. **Ocean masking** — NaN out all pixels with elevation ≤ 0 m (ocean/sea, located to the west of the area).

All data is accessed cloud-natively via the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/) STAC API — no downloads required.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import xarray as xr
import rioxarray as rxr
from rioxarray.merge import merge_arrays
import pystac_client
import planetary_computer
import hvplot.xarray

# Rakhine State / Cyclone Mocha landfall zone
BBOX = [90.9, 17.5, 94.6, 21.3]   # [lon_min, lat_min, lon_max, lat_max]

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Connected to Planetary Computer STAC API")

## 2. DTM — Copernicus GLO-30 vs GLO-90 comparison

The Copernicus Digital Elevation Model is available at two resolutions on Planetary Computer:
- **`cop-dem-glo-30`** — 30 m posting (~1 arc-second)
- **`cop-dem-glo-90`** — 90 m posting (~3 arc-second)

Loading both lets us compare the terrain detail trade-off.

In [ ]:
items_30 = catalog.search(collections=["cop-dem-glo-30"], bbox=BBOX).item_collection()
items_90 = catalog.search(collections=["cop-dem-glo-90"], bbox=BBOX).item_collection()
print(f"GLO-30 items: {len(items_30)}")
print(f"GLO-90 items: {len(items_90)}")

In [ ]:
def load_dem(items):
    """Open each DEM tile individually, merge them, then clip to BBOX."""
    tiles = []
    for item in items:
        href = item.assets["data"].href
        da = rxr.open_rasterio(href, chunks="auto").squeeze("band", drop=True)
        tiles.append(da)
    merged = merge_arrays(tiles)
    clipped = merged.rio.clip_box(*BBOX)
    return clipped.compute()

print("Loading GLO-30 tiles...")
dem_30 = load_dem(items_30)
print(f"GLO-30 shape: {dem_30.shape},  min={float(dem_30.min()):.1f} m,  max={float(dem_30.max()):.1f} m,  mean={float(dem_30.mean()):.1f} m")

print("Loading GLO-90 tiles...")
dem_90 = load_dem(items_90)
print(f"GLO-90 shape: {dem_90.shape},  min={float(dem_90.min()):.1f} m,  max={float(dem_90.max()):.1f} m,  mean={float(dem_90.mean()):.1f} m")

In [ ]:
crs_dem = dem_30.rio.crs.to_string() if dem_30.rio.crs else "EPSG:4326"

p30 = dem_30.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-30 (30 m)",
)

p90 = dem_90.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-90 (90 m)",
)

p30 + p90

## 3. Ocean Masking

The Bay of Bengal lies to the **west** of the study area.  
Copernicus DEM encodes ocean/sea pixels as zero or negative values.  
We mask all pixels with elevation **≤ 0 m** to `NaN`, producing land-only DEMs
(`dem_30_land` and `dem_90_land`) that can be used without the ocean dominating
colour scales or statistics.

In [ ]:
# Mask ocean: elevation <= 0 → NaN
dem_30_land = dem_30.where(dem_30 > 0)
dem_90_land = dem_90.where(dem_90 > 0)

print(f"GLO-30 land  —  min={float(dem_30_land.min()):.1f} m,  "
      f"max={float(dem_30_land.max()):.1f} m,  "
      f"mean={float(dem_30_land.mean()):.1f} m")
print(f"GLO-90 land  —  min={float(dem_90_land.min()):.1f} m,  "
      f"max={float(dem_90_land.max()):.1f} m,  "
      f"mean={float(dem_90_land.mean()):.1f} m")

In [ ]:
p30_land = dem_30_land.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-30 — land only (ocean = NaN)",
)

p90_land = dem_90_land.hvplot(
    x="x", y="y",
    cmap="terrain",
    clim=(0, 2000),
    crs=crs_dem,
    tiles="EsriImagery",
    rasterize=True,
    alpha=0.7,
    frame_width=420,
    colorbar=True,
    title="Copernicus DEM GLO-90 — land only (ocean = NaN)",
)

p30_land + p90_land